# 🤖 Smart Research Assistant Agent
### Built with LangChain + LangGraph | 100% FREE (OpenRouter + HuggingFace)

**What this agent does:**
- 📄 Searches your uploaded PDF using RAG
- 📝 Generates structured research reports (Executive Summary, Key Findings, Conclusion)
- 💰 Estimates LLM API call costs
- 🧠 Remembers conversation history within a session
- ⚡ Streams responses token by token

**Zero Cost Stack:**
| Component | Free Solution |
|-----------|---------------|
| LLM | OpenRouter (Llama 3.3 70B - FREE) |
| Embeddings | HuggingFace all-MiniLM-L6-v2 (local) |
| Vector DB | ChromaDB (in-memory) |
| Agent | LangGraph ReAct Agent |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## STEP 0 — Install All Libraries

In [ ]:
!pip install -q \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    langgraph \
    chromadb \
    pypdf \
    tiktoken \
    sentence-transformers \
    pydantic

print("✅ All libraries installed successfully!")

---
## STEP 1 — Configuration & Setup

> **Get your FREE API key:**
> 1. Go to https://openrouter.ai
> 2. Sign up (no credit card needed)
> 3. Go to https://openrouter.ai/keys
> 4. Create a key and paste it below

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════
# ⚙️  CONFIGURATION — Edit these values!
# ══════════════════════════════════════════════

OPENROUTER_API_KEY = "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"  # <-- Paste your key here

PDF_PATH = "/content/1-s2.0-S0264275124007480-main.pdf"
FREE_MODEL = "meta-llama/llama-3-8b-instruct"  # Best free all-rounder

# Other free options:
# "google/gemma-3-27b-it:free"
# "mistralai/mistral-small-3.1-24b-instruct:free"
# "deepseek/deepseek-r1:free"

# ══════════════════════════════════════════════
# Set environment variable
os.environ["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"] = OPENROUTER_API_KEY

print("✅ Configuration set!")
print(f"   Model  : {FREE_MODEL}")
print(f"   PDF    : {PDF_PATH}")

---
## STEP 2 — All Imports

In [ ]:
# ── LangChain Core ──
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool

# ── Document Loading & RAG ──
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# ── LangGraph Agent ──
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# ── Standard ──
import json
import os

print("✅ All imports successful!")

---
## STEP 3 — Initialize the FREE LLM

In [ ]:
# Initialize the LLM via OpenRouter (OpenAI-compatible API)
llm = ChatOpenAI(
    model=FREE_MODEL,
    temperature=0,                              # 0 = factual, consistent
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",   # OpenRouter endpoint
)

# Quick sanity test
test = llm.invoke("Reply with exactly: 'LLM ready!'")
print(f"LLM Test: {test.content}")
print(f"✅ LLM initialized: {FREE_MODEL}")

---
## STEP 4 — RAG Setup: Load PDF → Embed → Vector DB

> **What happens here:**
> PDF → Split into chunks → Convert to vectors → Store in ChromaDB → Ready to search!

In [ ]:
# ══════════════════════════════════════════════
# RAG SETUP — PDF → ChromaDB
# ══════════════════════════════════════════════
print("📄 Loading PDF...")

# Step 1: Load PDF
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"   Loaded {len(pages)} pages from '{PDF_PATH}'")

# Step 2: Split into chunks
# chunk_size=500: each chunk ~500 chars
# chunk_overlap=50: 50 chars overlap between chunks (preserves context at boundaries)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(pages)
print(f"   Split into {len(chunks)} chunks")

# Step 3: FREE Embeddings using HuggingFace (runs locally, no API needed!)
print("🧠 Loading embedding model (first time may take ~30 seconds)...")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",  # Lightweight but powerful
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("   Embedding model loaded!")

# Step 4: Build ChromaDB vector store
print("📦 Building vector database...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="research_docs"
)

# Step 5: Create retriever — returns top 3 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(f"\n✅ RAG setup complete!")
print(f"   {len(chunks)} chunks indexed and ready to search")

# Quick test
test_docs = retriever.invoke("main topic")
print(f"   Test retrieval: found {len(test_docs)} relevant chunks")

---
## STEP 5 — Build the 3 Custom Tools

> **Key concept:** The agent reads each tool's docstring to decide WHEN to use it.
> Write docstrings like instructions to the AI, not like comments for humans!

In [ ]:
# ══════════════════════════════════════════════
# TOOL 1: Search Documents (RAG Tool)
# ══════════════════════════════════════════════
@tool
def search_documents(query: str) -> str:
    """Search uploaded research documents for relevant information.
    Use this tool when the user asks about document content, wants to know
    what is in the PDF, or asks any question about the uploaded material.
    Input should be a search query string."""

    docs = retriever.invoke(query)

    if not docs:
        return "No relevant documents found for this query."

    results = []
    for doc in docs:
        source = doc.metadata.get("source", "?")
        page   = doc.metadata.get("page", "?")
        results.append(f"[{source} | Page {page}]:\n{doc.page_content}")

    return "\n\n".join(results)


# ══════════════════════════════════════════════
# TOOL 2: Generate Report (LCEL Chain inside!)
# ══════════════════════════════════════════════
@tool
def generate_report(topic: str, findings: str) -> str:
    """Generate a structured research report given a topic and findings.
    Use this tool when the user asks for a report, summary, structured output,
    or wants to turn research findings into a professional document.
    Requires: topic (what the report is about) and findings (raw content to structure)."""

    # LCEL chain: prompt | llm | parser
    report_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a professional research analyst.
        Write clear, structured reports with proper sections.
        Use markdown formatting for headers."""),
        ("human", """Write a structured research report on the topic: '{topic}'

Based on these findings:
{findings}

Structure your report with exactly these three sections:

## Executive Summary
(2-3 sentences overview)

## Key Findings
(bullet points of main insights)

## Conclusion
(final thoughts and implications)""")
    ])

    # LCEL pipe chain
    report_chain = report_prompt | llm | StrOutputParser()

    result = report_chain.invoke({
        "topic": topic,
        "findings": findings
    })

    return result


# ══════════════════════════════════════════════
# TOOL 3: Estimate Cost
# ══════════════════════════════════════════════
@tool
def estimate_cost(prompt_tokens: int, completion_tokens: int, model: str = "gpt-4o-mini") -> str:
    """Estimate the cost of LLM API calls given token counts.
    Use this tool when the user asks about cost, pricing, how much tokens cost,
    or wants to calculate the price of API calls.
    Inputs: prompt_tokens (int), completion_tokens (int), model (str, optional)."""

    # Pricing per 1 million tokens (as of 2024)
    pricing = {
        "gpt-4o-mini": {
            "input":  0.150,   # $0.150 per 1M input tokens
            "output": 0.600    # $0.600 per 1M output tokens
        },
        "gpt-4o": {
            "input":  2.50,
            "output": 10.00
        },
        "gpt-3.5-turbo": {
            "input":  0.50,
            "output": 1.50
        },
        "llama-3.3-70b": {
            "input":  0.00,    # FREE on OpenRouter!
            "output": 0.00
        }
    }

    # Normalize model name for lookup
    model_key = model.lower().strip()

    # Find matching model
    matched = None
    for key in pricing:
        if key in model_key or model_key in key:
            matched = key
            break

    if not matched:
        supported = ", ".join(pricing.keys())
        return f"Model '{model}' not found. Supported models: {supported}"

    # Calculate cost
    input_cost  = (prompt_tokens     / 1_000_000) * pricing[matched]["input"]
    output_cost = (completion_tokens / 1_000_000) * pricing[matched]["output"]
    total_cost  = input_cost + output_cost

    return (
        f"Cost Estimate for {matched}:\n"
        f"  Input  tokens : {prompt_tokens:,}   → ${input_cost:.6f}\n"
        f"  Output tokens : {completion_tokens:,}   → ${output_cost:.6f}\n"
        f"  ─────────────────────────────────\n"
        f"  Total cost    : ${total_cost:.6f}"
    )


# ══════════════════════════════════════════════
# Register all tools
# ══════════════════════════════════════════════
tools = [search_documents, generate_report, estimate_cost]

print("✅ All 3 tools created!")
for t in tools:
    print(f"   🔧 {t.name}")

---
## STEP 6 — Create the LangGraph ReAct Agent with Memory

In [ ]:
# ══════════════════════════════════════════════
# CREATE LANGGRAPH AGENT
# ══════════════════════════════════════════════

# Memory: stores conversation so agent remembers previous messages
memory = MemorySaver()

# Create the ReAct agent
# ReAct = Reason + Act
# Agent reads tool docstrings → reasons which tool to call → calls it → reasons again → answers
agent = create_react_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,         # Attach memory
)

print("✅ ReAct Agent created with:")
print(f"   🤖 Model    : {FREE_MODEL}")
print(f"   🔧 Tools    : {[t.name for t in tools]}")
print(f"   🧠 Memory   : MemorySaver (in-session)")
print(f"   📡 Streaming: Enabled")

---
## STEP 7 — The ask() Helper Function (with Streaming)

In [ ]:
# ══════════════════════════════════════════════
# PDF SETUP
# ══════════════════════════════════════════════
!pip install -q fpdf2

from fpdf import FPDF
import datetime
from IPython.display import display, FileLink
import io

def clean_text(text):
    """Unicode chars ko safe chars se replace karo."""
    replacements = {
        '\u2014': '-',    # em dash —
        '\u2013': '-',    # en dash –
        '\u2018': "'",    # left single quote '
        '\u2019': "'",    # right single quote '
        '\u201c': '"',    # left double quote "
        '\u201d': '"',    # right double quote "
        '\u2022': '*',    # bullet •
        '\u2026': '...',  # ellipsis …
        '\u00b0': ' degrees',
        '\u20b9': 'Rs',
        '\u2192': '->',   # arrow →
        '\u2190': '<-',   # arrow ←
        '\u2713': 'OK',   # checkmark ✓
        '\u274c': 'X',    # cross ❌
        '\u2705': 'OK',   # green tick ✅
        '\u2728': '*',    # sparkles ✨
    }
    for char, replacement in replacements.items():
        text = text.replace(char, replacement)
    return text.encode('latin-1', errors='replace').decode('latin-1')


class ResearchPDF(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 13)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 12, "Research Assistant - AI Generated Report",
                  fill=True, new_x="LMARGIN", new_y="NEXT", align="C")
        self.set_text_color(0, 0, 0)
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(150, 150, 150)
        self.cell(0, 10,
            f"Page {self.page_no()} | {datetime.datetime.now().strftime('%d %b %Y %H:%M')}",
            align="C")


all_qa = []
q_count = [0]
PDF_OUTPUT = "/content/research_output.pdf"


def rebuild_and_save_pdf():
    """
    Har baar fresh PDF banao saare stored Q&A se.
    output() close issue fix ho jaata hai!
    """
    fresh_pdf = ResearchPDF()
    fresh_pdf.set_auto_page_break(auto=True, margin=15)
    fresh_pdf.add_page()

    # Title
    fresh_pdf.set_font("Helvetica", "B", 11)
    fresh_pdf.set_text_color(41, 128, 185)
    fresh_pdf.cell(0, 8,
        "Document: Women's Perceived Safety in Public Places",
        new_x="LMARGIN", new_y="NEXT")
    fresh_pdf.set_font("Helvetica", "", 10)
    fresh_pdf.set_text_color(100, 100, 100)
    fresh_pdf.cell(0, 6,
        f"Session: {datetime.datetime.now().strftime('%d %B %Y, %H:%M')}",
        new_x="LMARGIN", new_y="NEXT")
    fresh_pdf.ln(5)

    for i, qa in enumerate(all_qa, 1):

        # Question box
        fresh_pdf.set_fill_color(235, 245, 255)
        fresh_pdf.set_font("Helvetica", "B", 11)
        fresh_pdf.set_text_color(41, 128, 185)
        q_text = clean_text(
            f"Q{i}: {qa['question'][:90]}{'...' if len(qa['question'])>90 else ''}"
        )
        fresh_pdf.cell(0, 9, q_text,
            fill=True, new_x="LMARGIN", new_y="NEXT")
        fresh_pdf.ln(2)

        # Answer
        fresh_pdf.set_font("Helvetica", "", 10)
        fresh_pdf.set_text_color(30, 30, 30)
        fresh_pdf.multi_cell(0, 6, clean_text(qa['answer']))
        fresh_pdf.ln(4)

        # Divider
        fresh_pdf.set_draw_color(200, 200, 200)
        fresh_pdf.line(10, fresh_pdf.get_y(), 200, fresh_pdf.get_y())
        fresh_pdf.ln(6)

    # Save
    fresh_pdf.output(PDF_OUTPUT)


# ══════════════════════════════════════════════
# ask() — Har answer PDF mein save hoga
# ══════════════════════════════════════════════
def ask(question: str, session_id: str = "research-session-1"):
    q_count[0] += 1
    qnum = q_count[0]

    config = {"configurable": {"thread_id": session_id}}
    full_response = ""

    for chunk in agent.stream(
        {"messages": [("human", question)]},
        config=config
    ):
        if "agent" in chunk:
            messages = chunk["agent"].get("messages", [])
            if messages:
                last_msg = messages[-1]
                if hasattr(last_msg, "content") and last_msg.content:
                    full_response += last_msg.content

    all_qa.append({
        "question": question,
        "answer": full_response.strip()
    })

    rebuild_and_save_pdf()

    print(f"✅ Q{qnum} saved to PDF!")
    display(FileLink(
        PDF_OUTPUT,
        result_html_prefix=f"📄 Download PDF (Q{qnum} added): "
    ))


print("✅ ask() ready — har answer PDF mein jayega!")
print(f"   Total questions so far: {q_count[0]}")


---
## STEP 8 — Test the Agent!

### Test 1: Document Search (RAG)

In [ ]:
# TEST 1: Document Search
ask("What factors affect women's perceived safety?")

### Test 2: Report Generation (RAG + LCEL Chain)

In [ ]:
# TEST 2: Report Generation
ask("Search the document and generate a structured report on built environment factors affecting women's safety.")

### Test 3: Cost Estimation

In [ ]:
# TEST 3: Cost Estimation
ask("How much would 5000 input tokens and 2000 output tokens cost using gpt-4o-mini?")

### Test 4: Memory Test (Most Important!)

In [ ]:
# TEST 4: Memory Test
ask("Add a Recommendations section to the report you just made.")

---
## ✅ Completion Checklist

| Check | Item |
|-------|------|
| ✅ | Libraries installed |
| ✅ | OpenRouter LLM initialized |
| ✅ | PDF loaded and chunked |
| ✅ | HuggingFace embeddings working |
| ✅ | ChromaDB vector store built |
| ✅ | Tool 1: search_documents (RAG) |
| ✅ | Tool 2: generate_report (LCEL chain inside) |
| ✅ | Tool 3: estimate_cost |
| ✅ | LangGraph ReAct Agent created |
| ✅ | MemorySaver attached |
| ✅ | ask() streaming function working |
| ✅ | Test 1: PDF content returned with [source] |
| ✅ | Test 2: Report has 3 sections |
| ✅ | Test 3: Dollar cost returned |
| ✅ | Test 4: Memory working across questions |

---
## 🎁 BONUS — Interactive Chat Loop

In [ ]:
# ══════════════════════════════════════════════
# BONUS: Interactive Chat with the Agent
# Type 'exit' to quit
# ══════════════════════════════════════════════
print("🤖 Research Assistant ready! Type 'exit' to quit.")
print("Available capabilities:")
print("  • Ask about document content")
print("  • Request a research report")
print("  • Ask about API costs")
print("─" * 60)

while True:
    try:
        user_input = input("\n🧑 You: ").strip()
        if user_input.lower() in ["exit", "quit", "bye"]:
            print("👋 Goodbye!")
            break
        if not user_input:
            continue
        ask(user_input, session_id="interactive-session")
    except KeyboardInterrupt:
        print("\n👋 Interrupted. Goodbye!")
        break